# Sequential Multi-Agent Pattern

The **sequential multi-agent pattern** runs specialists in a **fixed order**: Agent A completes, passes output to Agent B, then Agent C. The orchestrator does not debate or dynamically route — it walks a **pipeline**.

```
START ──► Researcher ──► Writer ──► Critic ──► END
```

This is the simplest multi-agent topology and the most **auditable**. It fits content pipelines, ETL-with-LLM stages, and any workflow where the steps are known upfront.

This notebook implements a production-shaped **ReleaseFlow** sequential pipeline in plain Python: shared state, hand-off contracts, optional review loop, per-stage testing, tracing, and error handling. No frameworks or prior notebooks are required.

In [ ]:
"""
ReleaseFlow — sequential multi-agent pipeline for v2.4.0 release announcements.
"""

import json
import os
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


CHANGELOG: list[dict[str, str]] = [
    {"version": "2.4.0", "item": "Added bulk export endpoint for reports"},
    {"version": "2.4.0", "item": "Fixed session timeout on mobile browsers"},
    {"version": "2.4.0", "item": "Deprecated legacy v1 webhook format"},
]

RUNBOOKS: list[dict[str, str]] = [
    {"id": "comm-201", "title": "Customer communication standards",
     "text": "Lead with user benefit, list breaking changes separately, include support channel #releases."},
]

STYLE_GUIDE = {
    "must_include": ["version number", "breaking changes section", "support channel"],
    "max_words": 200,
}

PUBLISHED: list[dict[str, Any]] = []

print(f"ReleaseFlow corpus: {len(CHANGELOG)} changelog items")

---

## 1. Pattern Definition

| Property | Sequential pattern |
|----------|------------------|
| **Topology** | Linear chain A → B → C |
| **Routing** | Fixed at design time |
| **Communication** | Shared state / artifacts passed forward |
| **Concurrency** | One agent active at a time |
| **Determinism** | Highest among multi-agent patterns |

**Formal shape:**

```
state_0 ──► agent_1(state_0) ──► state_1 ──► agent_2(state_1) ──► ... ──► state_n
```

Each agent is a **pure function over shared state** (in the ideal case): read artifacts, write artifacts, append logs. The orchestrator only controls **order**, not **content**.

---

## 2. When to Use Sequential — and When Not To

| Use sequential when | Avoid sequential when |
|---------------------|----------------------|
| Steps are always the same order | Path depends on runtime discovery |
| Each stage has a clear input/output contract | Agents need to debate or negotiate |
| Compliance requires auditable step order | Independent subtasks can run in parallel |
| You want simple integration tests per stage | One generalist agent suffices |

**ReleaseFlow fit:** Research → Write → Review is a classic pipeline. Review may **loop back** to Write, but that is still a **conditional edge on a mostly linear graph** — not dynamic speaker selection.

---

## 3. Pipeline with Optional Review Loop

```
                    ┌─────────────┐
                    │  Researcher │
                    └──────┬──────┘
                           ▼
                    ┌─────────────┐
              ┌────►│   Writer    │◄────┐
              │     └──────┬──────┘     │ reject (retry < max)
              │            ▼            │
              │     ┌─────────────┐       │
              └─────│   Critic    │───────┘
                    └──────┬──────┘
                           │ approve
                           ▼
                    ┌─────────────┐
                    │  Publisher  │
                    └─────────────┘
```

The **back-edge** (critic → writer) is still orchestrator-owned code — not emergent chat routing.

---

## 4. Shared State and Hand-off Contracts

Sequential pipelines fail when stages **assume artifacts that do not exist**. Define contracts explicitly:

| Stage | Reads | Writes |
|-------|-------|--------|
| `researcher` | `goal` | `research` |
| `writer` | `research`, `review_feedback` (optional) | `draft` |
| `critic` | `draft` | `approved`, `review_feedback` |
| `publisher` | `draft`, `approved` | `announcement_id` |

In [ ]:
@dataclass
class PipelineState:
    run_id: str
    goal: str
    artifacts: dict[str, Any] = field(default_factory=dict)
    messages: list[dict[str, str]] = field(default_factory=list)
    status: str = "running"  # running | approved | rejected | published | failed
    revision_count: int = 0

    def log(self, agent: str, content: str) -> None:
        self.messages.append({
            "agent": agent,
            "content": content,
            "ts": datetime.now(timezone.utc).isoformat(),
        })


STEP_CONTRACTS: dict[str, dict[str, list[str]]] = {
    "researcher": {"reads": ["goal"], "writes": ["research"]},
    "writer": {"reads": ["research"], "writes": ["draft"]},
    "critic": {"reads": ["draft"], "writes": ["approved", "review_feedback"]},
    "publisher": {"reads": ["draft", "approved"], "writes": ["announcement_id"]},
}


def validate_contract(stage: str, state: PipelineState) -> list[str]:
    """Return missing artifact keys required before stage runs."""
    contract = STEP_CONTRACTS.get(stage, {})
    missing = []
    for key in contract.get("reads", []):
        if key == "goal":
            continue
        if key == "review_feedback":
            continue  # optional on first write
        if key not in state.artifacts:
            missing.append(key)
    return missing


demo = PipelineState(run_id="demo", goal="test")
print("writer before research:", validate_contract("writer", demo))
demo.artifacts["research"] = {}
print("writer after research:", validate_contract("writer", demo))

---

## 5. Specialist Agents

Each agent implements `run(state) → state`. In production these wrap LLM calls; here they use deterministic logic so the pipeline runs offline.

In [ ]:
class ResearcherAgent:
    name = "researcher"

    def run(self, state: PipelineState) -> PipelineState:
        facts = {
            "version": "2.4.0",
            "changes": [c["item"] for c in CHANGELOG],
            "comm_standards": RUNBOOKS[0]["text"],
        }
        state.artifacts["research"] = facts
        state.log(self.name, f"Gathered {len(facts['changes'])} changelog items for v{facts['version']}")
        return state


class WriterAgent:
    name = "writer"

    def run(self, state: PipelineState) -> PipelineState:
        facts = state.artifacts.get("research", {})
        feedback = state.artifacts.get("review_feedback", "")
        draft = (
            f"Release v{facts.get('version', '?')} is now available.\n\n"
            f"What's new: " + "; ".join(facts.get("changes", [])) + ".\n\n"
            f"Breaking changes: Legacy v1 webhook format is deprecated.\n\n"
            f"Questions? Contact #releases."
        )
        if feedback and "too long" in feedback:
            draft = "\n".join(draft.split("\n")[:4]) + "\n\nQuestions? Contact #releases."
        state.artifacts["draft"] = draft
        state.log(self.name, f"Draft ready ({len(draft.split())} words)")
        return state


class CriticAgent:
    name = "critic"

    def run(self, state: PipelineState) -> PipelineState:
        draft = state.artifacts.get("draft", "")
        issues = []
        if "v2.4" not in draft:
            issues.append("missing version number")
        if "Breaking changes" not in draft:
            issues.append("missing breaking changes section")
        if "#releases" not in draft:
            issues.append("missing support channel")
        if len(draft.split()) > STYLE_GUIDE["max_words"]:
            issues.append(f"too long ({len(draft.split())} words)")

        if issues:
            state.artifacts["approved"] = False
            state.artifacts["review_feedback"] = ", ".join(issues)
            state.status = "rejected"
            state.log(self.name, f"REJECTED: {state.artifacts['review_feedback']}")
        else:
            state.artifacts["approved"] = True
            state.artifacts["review_feedback"] = ""
            state.status = "approved"
            state.log(self.name, "APPROVED: meets style guide")
        return state


class PublisherAgent:
    name = "publisher"

    def run(self, state: PipelineState) -> PipelineState:
        if not state.artifacts.get("approved"):
            state.status = "failed"
            state.log(self.name, "Blocked — draft not approved")
            return state
        ann = {
            "id": f"ANN-{uuid.uuid4().hex[:6].upper()}",
            "body": state.artifacts.get("draft", ""),
            "published_at": datetime.now(timezone.utc).isoformat(),
        }
        PUBLISHED.append(ann)
        state.artifacts["announcement_id"] = ann["id"]
        state.status = "published"
        state.log(self.name, f"Published {ann['id']}")
        return state


AGENT_REGISTRY: dict[str, Any] = {
    "researcher": ResearcherAgent(),
    "writer": WriterAgent(),
    "critic": CriticAgent(),
    "publisher": PublisherAgent(),
}
print("Agents:", list(AGENT_REGISTRY.keys()))

---

## 6. SequentialPipeline Orchestrator

The orchestrator walks a **step list**, validates contracts, logs traces, and handles the review loop.

In [ ]:
@dataclass
class PipelineStep:
    name: str
    agent_name: str


@dataclass
class StepTrace:
    index: int
    step: str
    agent: str
    status: str
    ms: float
    missing_contract: list[str] = field(default_factory=list)


@dataclass
class PipelineResult:
    run_id: str
    final_status: str
    approved: bool
    published: bool
    revision_count: int
    trace: list[StepTrace]
    draft_preview: str = ""
    announcement_id: str = ""


class SequentialPipeline:
    """Fixed-order multi-agent pipeline with optional write↔review loop."""

    def __init__(self, max_revisions: int = 2) -> None:
        self.max_revisions = max_revisions
        self.linear_steps = [
            PipelineStep("gather_facts", "researcher"),
        ]
        self.review_loop = [
            PipelineStep("draft", "writer"),
            PipelineStep("review", "critic"),
        ]
        self.tail_steps = [
            PipelineStep("publish", "publisher"),
        ]

    def _run_step(self, step: PipelineStep, state: PipelineState, trace: list[StepTrace], idx: int) -> PipelineState:
        missing = validate_contract(step.agent_name, state)
        t0 = datetime.now(timezone.utc)
        if missing:
            trace.append(StepTrace(idx, step.name, step.agent_name, "contract_error", 0, missing))
            state.status = "failed"
            state.log("orchestrator", f"Contract fail at {step.name}: missing {missing}")
            return state
        agent = AGENT_REGISTRY[step.agent_name]
        state = agent.run(state)
        ms = (datetime.now(timezone.utc) - t0).total_seconds() * 1000
        trace.append(StepTrace(idx, step.name, step.agent_name, "ok", round(ms, 2)))
        return state

    def run(self, goal: str, *, force_long_draft: bool = False) -> PipelineResult:
        state = PipelineState(run_id=f"run_{uuid.uuid4().hex[:8]}", goal=goal)
        trace: list[StepTrace] = []
        step_idx = 0

        for step in self.linear_steps:
            step_idx += 1
            state = self._run_step(step, state, trace, step_idx)
            if state.status == "failed":
                break

        while state.status != "failed":
            for step in self.review_loop:
                step_idx += 1
                state = self._run_step(step, state, trace, step_idx)
                if state.status == "failed":
                    break
                if step.agent_name == "writer" and force_long_draft and state.revision_count == 0:
                    state.artifacts["draft"] = state.artifacts.get("draft", "") + " extra" * 60
            if state.status == "failed":
                break
            if state.artifacts.get("approved"):
                break
            if state.revision_count >= self.max_revisions:
                state.log("orchestrator", f"Max revisions ({self.max_revisions}) reached")
                break
            state.revision_count += 1
            state.log("orchestrator", f"Revision loop {state.revision_count}: writer ← critic")

        if state.status != "failed" and state.artifacts.get("approved"):
            for step in self.tail_steps:
                step_idx += 1
                state = self._run_step(step, state, trace, step_idx)
                if state.status == "failed":
                    break

        return PipelineResult(
            run_id=state.run_id,
            final_status=state.status,
            approved=bool(state.artifacts.get("approved")),
            published=state.status == "published",
            revision_count=state.revision_count,
            trace=trace,
            draft_preview=state.artifacts.get("draft", "")[:200],
            announcement_id=state.artifacts.get("announcement_id", ""),
        )


pipeline = SequentialPipeline(max_revisions=2)
print("SequentialPipeline ready")

---

## 7. Happy-Path Run

Research → Write → Review (approved) → Publish.

In [ ]:
happy = pipeline.run("Produce v2.4.0 customer release announcement")

print(f"Run: {happy.run_id}")
print(f"Status: {happy.final_status} | Approved: {happy.approved} | Published: {happy.published}")
print(f"Announcement: {happy.announcement_id}")
print("\nPipeline trace:")
for t in happy.trace:
    print(f"  {t.index}. {t.step} ({t.agent}) — {t.status} [{t.ms}ms]")
print(f"\nDraft preview:\n{happy.draft_preview}...")

---

## 8. Review Loop — Revision Demo

Force an overlong draft so the critic rejects once; the pipeline loops writer → critic, then publishes.

In [ ]:
retry = pipeline.run("v2.4.0 with revision", force_long_draft=True)

print(f"Revisions: {retry.revision_count} | Published: {retry.published}")
agents_run = [t.agent for t in retry.trace]
print(f"Agent order: {' → '.join(agents_run)}")
print(f"Writer passes: {agents_run.count('writer')} | Critic passes: {agents_run.count('critic')}")

---

## 9. Testing Stages in Isolation

Sequential pipelines are **testable** because each stage has a narrow contract. Build minimal state fixtures per agent:

In [ ]:
def test_writer_with_fixture() -> dict[str, Any]:
    state = PipelineState(run_id="test", goal="test")
    state.artifacts["research"] = {
        "version": "2.4.0",
        "changes": ["Feature X"],
        "comm_standards": "Be concise.",
    }
    state = WriterAgent().run(state)
    return {
        "has_draft": "draft" in state.artifacts,
        "mentions_version": "v2.4" in state.artifacts.get("draft", ""),
        "word_count": len(state.artifacts.get("draft", "").split()),
    }


def test_critic_rejects_long_draft() -> dict[str, Any]:
    state = PipelineState(run_id="test", goal="test")
    state.artifacts["draft"] = "Release v2.4.0. " + "word " * 250 + " Breaking changes: x. #releases"
    state = CriticAgent().run(state)
    return {"approved": state.artifacts.get("approved"), "feedback": state.artifacts.get("review_feedback", "")}


def test_publisher_blocks_unapproved() -> dict[str, Any]:
    state = PipelineState(run_id="test", goal="test")
    state.artifacts["draft"] = "short draft"
    state.artifacts["approved"] = False
    state = PublisherAgent().run(state)
    return {"status": state.status, "published_id": state.artifacts.get("announcement_id")}


print("Writer fixture:", test_writer_with_fixture())
print("Critic long draft:", test_critic_rejects_long_draft())
print("Publisher gate:", test_publisher_blocks_unapproved())

---

## 10. Structured Trace Export

Production pipelines emit **machine-readable** traces for dashboards and audits.

In [ ]:
def trace_to_json_lines(result: PipelineResult) -> str:
    lines = []
    for t in result.trace:
        lines.append(json.dumps({
            "run_id": result.run_id,
            "step": t.index,
            "name": t.step,
            "agent": t.agent,
            "status": t.status,
            "ms": t.ms,
            "missing": t.missing_contract,
        }))
    lines.append(json.dumps({
        "run_id": result.run_id,
        "event": "pipeline_complete",
        "final_status": result.final_status,
        "revisions": result.revision_count,
    }))
    return "\n".join(lines)


print(trace_to_json_lines(happy))

---

## 11. Error Handling — Contract Violations

If a stage runs without required inputs, fail **fast** with a clear contract error — do not let an agent hallucinate missing context.

In [ ]:
class BrokenPipeline(SequentialPipeline):
    """Skips researcher to demonstrate contract failure."""

    def __init__(self) -> None:
        super().__init__()
        self.linear_steps = []  # skip research


broken = BrokenPipeline().run("contract failure demo")
print(f"Status: {broken.final_status}")
for t in broken.trace:
    if t.status == "contract_error":
        print(f"  Contract error at {t.step}: missing {t.missing_contract}")

---

## 12. Sequential vs Other Patterns

| Pattern | Routing | vs Sequential |
|---------|---------|---------------|
| **Sequential** | Fixed order | Baseline — simplest |
| **Group chat** | Manager / round-robin | More flexible, less auditable |
| **Supervisor** | Router each turn | Adaptive, extra LLM calls |
| **Parallel fan-out** | Concurrent branches | Faster when stages are independent |

Choose sequential when **order is law**. Add a supervisor or chat layer only when intake truly varies.

---

## 13. Idempotency and Side Effects

The **publisher** step has a **side effect** (writes to `PUBLISHED`). In production:

| Concern | Mitigation |
|---------|------------|
| Retry duplicates announcement | Idempotency key on `run_id` |
| Partial failure after publish | Two-phase: draft approved → confirm publish |
| Replay from checkpoint | Store `announcement_id` in state before marking published |

Earlier stages (research, write, review) should be **read-only** or write only to internal artifacts.

In [ ]:
class IdempotentPublisher(PublisherAgent):
    """Skip publish if this run_id already published."""

    def run(self, state: PipelineState) -> PipelineState:
        existing = state.artifacts.get("announcement_id")
        if existing:
            state.log(self.name, f"Idempotent skip — already published {existing}")
            state.status = "published"
            return state
        return super().run(state)


AGENT_REGISTRY["publisher"] = IdempotentPublisher()
state = PipelineState(run_id="idem_1", goal="test")
state.artifacts["research"] = {"version": "2.4.0", "changes": [], "comm_standards": ""}
state.artifacts["draft"] = "Release v2.4.0. Breaking changes: none. #releases"
state.artifacts["approved"] = True
state = IdempotentPublisher().run(state)
ann_id = state.artifacts["announcement_id"]
state = IdempotentPublisher().run(state)  # second call
print(f"Same announcement_id after retry: {state.artifacts.get('announcement_id') == ann_id}")

---

## 14. Production Checklist

1. **Document** read/write contract per stage.
2. **Validate** contracts before each agent runs.
3. **Cap** review loops (`max_revisions`).
4. **Trace** every step with latency and status.
5. **Unit test** each agent with fixtures — not only end-to-end.
6. **Gate** side effects (publish, email, charge) behind approval flags.
7. **Idempotency** keys on write steps that touch external systems.
8. **Version** prompts and schemas per stage independently.

---

## 15. Optional Live LLM — Writer Stage Only

Swap one pipeline stage for a live model without changing orchestration order.

In [ ]:
USE_LIVE_LLM = False


class LiveWriterAgent(WriterAgent):
    def run(self, state: PipelineState) -> PipelineState:
        if not USE_LIVE_LLM:
            return super().run(state)
        facts = state.artifacts.get("research", {})
        feedback = state.artifacts.get("review_feedback", "")
        prompt = (
            f"Write a customer release announcement for v{facts.get('version')}.\n"
            f"Changes: {facts.get('changes')}\nFeedback: {feedback or 'none'}\n"
            f"Include Breaking changes section and #releases. Max 200 words."
        )
        try:
            from openai import OpenAI
            client = OpenAI(api_key=OPENAI_API_KEY)
            resp = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
            )
            state.artifacts["draft"] = resp.choices[0].message.content or ""
        except Exception as exc:
            state.artifacts["draft"] = f"[LLM error: {exc}]"
        state.log(self.name, f"Live draft ({len(state.artifacts['draft'].split())} words)")
        return state


if USE_LIVE_LLM:
    AGENT_REGISTRY["writer"] = LiveWriterAgent()
    live_result = SequentialPipeline().run("Live writer demo")
    print(live_result.draft_preview)
else:
    print("Offline mode — set USE_LIVE_LLM = True to use OpenAI for the writer stage only.")
    print(f"Pipeline stages: researcher → writer → critic → publisher")

---

## 16. Anti-Patterns

| Anti-pattern | Problem | Fix |
|--------------|---------|-----|
| Hidden order in agent prompts | "Call writer next" inside critic | Orchestrator owns order |
| Skipping contract validation | Empty drafts from missing research | `validate_contract` before run |
| Unbounded review loop | Cost explosion | `max_revisions` |
| One giant agent doing all stages | Defeats purpose of pipeline | Split by role |
| Publishing without approval flag | Compliance incident | Publisher checks `approved` |

---

## 17. Quiz

1. Who decides execution order in a sequential pipeline?
2. What is a hand-off contract between stages?
3. Why cap review loops?
4. Which stage should be idempotent in ReleaseFlow?
5. When should you choose sequential over group chat?

---

## 18. Summary

| Concept | Takeaway |
|---------|----------|
| Sequential pattern | Fixed A → B → C agent order |
| Orchestrator | Owns order and conditional back-edges |
| Contracts | Each stage declares reads and writes |
| Review loop | Writer ↔ critic with `max_revisions` |
| Testing | Fixture state per agent, plus E2E pipeline |
| Traces | JSON-lines per step for audit |
| Side effects | Gate and idempotency on publisher |

Sequential is the **default** multi-agent pattern when your process is a pipeline. Master it before adding supervisor routing or group chat complexity.